In [1]:
from docx import Document
from docx.shared import Inches
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import json
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import confusion_matrix

## **Load dataset**

In [3]:
df = pd.read_csv('train_data.csv')
df["full_review"] = df["reviews.title"].fillna("") + " " + df["reviews.text"].fillna("")

## **Graph 1: `Sentiment Distribution`**


In [5]:
plt.figure()
df["sentiment"].value_counts().plot(kind="bar")
plt.title("Sentiment Distribution")
sentiment_chart = "sentiment_distribution.png"
plt.savefig(sentiment_chart)
plt.close()

## **Graph 2: `Review Length Distribution`**


In [7]:
df["review_length"] = df["full_review"].apply(len)

plt.figure()
plt.hist(df["review_length"], bins=50)
plt.title("Review Length Distribution")
length_chart = "review_length.png"
plt.savefig(length_chart)
plt.close()

## **Model for Confusion Matrix**


In [9]:
X = df["full_review"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(max_features=10000, stop_words="english")
X_train = tfidf.fit_transform(X_train)
X_test = tfidf.transform(X_test)

model = LinearSVC(class_weight="balanced")
model.fit(X_train, y_train)
pred = model.predict(X_test)

cm = confusion_matrix(y_test, pred)

plt.figure()
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.colorbar()
conf_matrix_chart = "confusion_matrix.png"
plt.savefig(conf_matrix_chart)
plt.close()

## **Creating Report**


In [11]:
doc = Document()
doc.add_heading("Amazon Product Sentiment Analysis Project", 0)

doc.add_paragraph("""
This project analyzes customer reviews of Amazon products to predict sentiment categories 
(Positive, Negative, Neutral). Various machine learning and deep learning techniques were 
implemented to address class imbalance and improve prediction accuracy.
""")

doc.add_heading("Exploratory Data Analysis", level=1)
doc.add_paragraph("Sentiment Distribution:")
doc.add_picture(sentiment_chart, width=Inches(5))

doc.add_paragraph("Review Length Distribution:")
doc.add_picture(length_chart, width=Inches(5))

doc.add_heading("Model Performance", level=1)
doc.add_paragraph("Confusion Matrix of Best Model (Linear SVM):")
doc.add_picture(conf_matrix_chart, width=Inches(5))

doc.add_heading("Conclusion", level=1)
doc.add_paragraph("""
The Linear SVM model achieved the best performance due to its effectiveness in handling 
high-dimensional textual data and class imbalance. Deep learning models such as LSTM 
were also explored for comparison.
""")

report_path = "Final_Report.docx"
doc.save(report_path)